<a href="https://colab.research.google.com/github/Mehroz485/ML-01/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mehroz485/ML-01/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Answer.** One row = one content item (`content_hash_id`), for one client (`client_hash_id`),
on one report date (`report_date`), inside `fact_content_daily_performance`. That's the row grain
I verify below, and it matches my lane's decision grain: Refresh / Content Opportunity Scoring
ranks *content items*, not client-days or queries, so a content-item-per-day fact table is the
right base to aggregate up from.

Time window: I develop everything on the mid-panel partition **`month=2026-03`**
(2026-03-01 -> 2026-03-31), split into an **early half** (days 1-15, my "known" window — this
is what a real refresh decision could see) and a **late half** (days 16-31, the "outcome"
window I'm not allowed to look at while building features). I never touch
`fact_content_daily_performance_sample` (June 2026) while developing — per the brief, that's
the sealed test month and the natural answer key for any past->future label.

In [1]:
# --- One-time setup for this Colab session ---
%pip -q install duckdb huggingface_hub pandas scikit-learn

import os, getpass
import duckdb

# Never hardcode the token -- this repo is public. Colab Secrets (key icon) or getpass only.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'  # mid-panel month -- cheap to iterate on, never the _sample month
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

print('Connected. Target partition:', f'fact_content_daily_performance/month={MONTH}')
# This cell only wires up the connection -- the three verification queries live in Section 3,
# each one directly under the claim it proves.


Paste your Hugging Face READ token (hf_...): ··········
Connected. Target partition: fact_content_daily_performance/month=2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Answer.**

- **Feature** (knowable *before* the decision point — day 16 of March): `gsc_impressions`,
  `gsc_clicks`, `gsc_avg_position`, all aggregated over days 1-15 only (the `*_early` columns
  built in Section 3). Nothing from days 16-31 ever enters the feature frame.
- **Label / proxy**: `is_declining` — 1 if late-window impressions (days 16-31) fall more than
  20% below the early-window impressions, else 0. Built only from `gsc_impressions` totals;
  never itself used as a feature (that's exactly the trap in Section 3).
- **Context** (grouping/joining only, never learned from): `client_hash_id`, `content_hash_id`,
  `report_date`.
- **Excluded, with why**:
  1. `fact_content_daily_performance_sample` (June 2026) — the final month. Using it to develop
     label logic means training on the answer key, per the brief's warning.
  2. GA4 columns (`ga4_data_available` and friends) — excluded as *features* this week. I use
     `ga4_data_available` once below purely to demonstrate the `IS TRUE` availability check;
     GA4 columns are `NULL`-heavy (not just TRUE/FALSE) and deserve their own contract pass
     before I'd trust them in a feature frame.

In [2]:
# Single source of truth for the bucket assignment above -- reused below so the
# code and the contract text can never quietly drift apart.
FEATURE_COLS  = ["imp_early", "clk_early", "ctr_early", "pos_early", "days_active_early"]
LABEL_COL     = "is_declining"
CONTEXT_COLS  = ["client_hash_id", "content_hash_id", "report_date"]
EXCLUDED = {
    "fact_content_daily_performance_sample": "final month (June 2026) -- sealed test window",
    "ga4_data_available / GA4 columns":       "not verified as features this week; used once "
                                               "below only to demo the IS TRUE availability check",
}
print("Features:", FEATURE_COLS)
print("Label:", LABEL_COL)
print("Excluded:", EXCLUDED)


Features: ['imp_early', 'clk_early', 'ctr_early', 'pos_early', 'days_active_early']
Label: is_declining
Excluded: {'fact_content_daily_performance_sample': 'final month (June 2026) -- sealed test window', 'ga4_data_available / GA4 columns': 'not verified as features this week; used once below only to demo the IS TRUE availability check'}


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three claims, three queries, in order: **grain** (Section 1's claim), **row count + date span**
(this slice, this month), and **availability** (`ga4_data_available`, filtered with `IS TRUE`
per the flyrank-data skill — the flag is NULL for millions of rows, not just TRUE/FALSE, so a
plain `= FALSE` or `NOT flag` silently miscounts). Then the five-feature frame, then the trap.

**Query 1 — grain.** `report_date + client_hash_id + content_hash_id` should be unique. Zero rows back means the grain holds.

In [3]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT_MONTH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Duplicate (report_date, client_hash_id, content_hash_id) combos found: {len(grain_check)}")
assert len(grain_check) == 0, "Grain claim from Section 1 is WRONG -- stop and re-check before building features."
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (report_date, client_hash_id, content_hash_id) combos found: 0


,report_date,client_hash_id,content_hash_id,n


**Query 2 — row count + date span.** This slice's size and coverage for `month=2026-03`, checked against what the docs promise (~78.8M rows across the *whole* daily fact; this is one month's slice of it).

In [4]:
counts = con.sql(f"""
    SELECT COUNT(*)                       AS n_rows,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date)                AS min_date,
           MAX(report_date)                AS max_date
    FROM {FACT_MONTH}
""").df()
counts

,n_rows,n_clients,n_content_items,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


**Query 3 — availability, `IS TRUE`.** How many rows survive the GA4 availability filter, and how many are TRUE / FALSE / NULL separately (this is the check the skill warns about — `NULL` is neither TRUE nor FALSE).

In [5]:
availability = con.sql(f"""
    SELECT COUNT(*)                                                     AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE  THEN 1 ELSE 0 END) AS rows_ga4_true,
           SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS rows_ga4_false,
           SUM(CASE WHEN ga4_data_available IS NULL  THEN 1 ELSE 0 END) AS rows_ga4_null,
           ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
                 / COUNT(*), 1)                                         AS pct_survive_is_true
    FROM {FACT_MONTH}
""").df()
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_ga4_true,rows_ga4_false,rows_ga4_null,pct_survive_is_true
0,9841378,413966.0,6408671.0,3018741.0,4.2


### Five features (max)

Built from the **early window only** (days 1-15 of March) — the point in time a real "should we
refresh this?" decision would actually be made. Each feature's "available when?" line:

1. **`imp_early`** — impressions summed over days 1-15. *Knowable at the decision moment because it's a sum of events that already happened before day 16.*
2. **`clk_early`** — clicks summed over days 1-15. *Same reasoning — fully observed history, no future window touched.*
3. **`ctr_early`** — `clk_early / imp_early`. *Derived arithmetically from two early-window columns; introduces no new information from later dates.*
4. **`pos_early`** — mean `gsc_avg_position` over days 1-15, positions of `0` (= "no data", not rank zero) excluded. *Same window as above, and the zero-guard keeps a documented data gotcha from corrupting the average.*
5. **`days_active_early`** — count of distinct days in 1-15 with `gsc_impressions > 0`. *A consistency signal built purely from which early days had any activity at all.*

In [6]:
features = con.sql(f"""
    WITH early AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)                                          AS imp_early,
               SUM(gsc_clicks)                                               AS clk_early,
               AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_early,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_active_early
        FROM {FACT_MONTH}
        WHERE report_date <= DATE '{MONTH}-15'
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 20   -- drop near-zero-volume pages, same spirit as notebook 03's HAVING guard
    ),
    late AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_late
        FROM {FACT_MONTH}
        WHERE report_date > DATE '{MONTH}-15'
        GROUP BY 1, 2
    )
    SELECT e.*, COALESCE(l.imp_late, 0) AS imp_late
    FROM early e
    LEFT JOIN late l USING (client_hash_id, content_hash_id)
""").df()

features["ctr_early"] = features["clk_early"] / features["imp_early"]
features[LABEL_COL] = (features["imp_late"] < 0.8 * features["imp_early"]).astype(int)

print(f"{len(features):,} content items with enough early-window volume (month={MONTH})")
print(features[LABEL_COL].value_counts(normalize=True).rename("share"))
features[CONTEXT_COLS[:2] + FEATURE_COLS + [LABEL_COL]].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

109,592 content items with enough early-window volume (month=2026-03)
is_declining
0    0.708911
1    0.291089
Name: share, dtype: float64


,client_hash_id,content_hash_id,imp_early,clk_early,ctr_early,pos_early,days_active_early,is_declining
0,client_3ffa76342f366962,content_f4464a7e2e5adfc6,82.0,0.0,0.000000,4.279471,15,0
1,client_3ffa76342f366962,content_15cd4e6402dbf7ba,27.0,0.0,0.000000,4.877381,8,1
2,client_3ffa76342f366962,content_fd41b905fa6f913a,30.0,2.0,0.066667,2.900000,13,0
3,client_3ffa76342f366962,content_4ba413a3527f8227,25.0,0.0,0.000000,7.527778,6,1
4,client_e547b89c05043229,content_d0fa1bbfbc10caf8,826.0,1.0,0.001211,22.301876,13,0


### The trap — the leakage lesson from notebook 02, on real warehouse data

Train honest first, on the five features only. Then deliberately add **one column built from the
outcome window** (`imp_late`, exactly what the label is made from) as a "feature," watch the
score jump toward perfect, then delete it and keep the honest number.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

model_df = features.dropna(subset=FEATURE_COLS + [LABEL_COL]).copy()
X = model_df[FEATURE_COLS]
y = model_df[LABEL_COL]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])

print(f"HONEST AUC -- 5 features, no leakage: {honest_auc:.3f}")

HONEST AUC -- 5 features, no leakage: 0.597


In [8]:
# --- Deliberately break the rule: add a column built directly from the outcome window ---
model_df["pct_change_leak"] = (model_df["imp_late"] - model_df["imp_early"]) / model_df["imp_early"]
# is_declining == (pct_change_leak < -0.2) almost by construction -- this smuggles the label in.

leak_cols = FEATURE_COLS + ["pct_change_leak"]
X_leak = model_df[leak_cols]

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)
leak_model = LogisticRegression(max_iter=1000).fit(X_tr_l, y_tr_l)
leak_auc = roc_auc_score(y_te_l, leak_model.predict_proba(X_te_l)[:, 1])

print(f"LEAKED AUC -- 6 \"features\", label smuggled in via pct_change_leak: {leak_auc:.3f}")
print(f"Jump vs honest: {leak_auc - honest_auc:+.3f}")

LEAKED AUC -- 6 "features", label smuggled in via pct_change_leak: 1.000
Jump vs honest: +0.403


In [9]:
# Delete the leaked column. Keep the honest number -- that is the whole lesson.
del model_df["pct_change_leak"]

print(f"Leaked column dropped from model_df.")
print(f"Honest AUC (kept):  {honest_auc:.3f}")
print(f"Leaked AUC (discarded, never to be reported as a real result): {leak_auc:.3f}")

Leaked column dropped from model_df.
Honest AUC (kept):  0.597
Leaked AUC (discarded, never to be reported as a real result): 1.000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation.** This slice is GSC-only and within-month: the early/late split (15 vs 16
days) is far shorter than the 90-day trend windows used elsewhere in this project, so
`is_declining` here is noisier than a true longer-horizon label and shouldn't be read as the
final capstone target as-is — it's a proxy built to prove the leakage lesson on real data, per
this week's brief. It also inherits the panel's known gap: history depth differs per client, and
this slice can't yet tell "zero impressions because nothing happened" apart from "zero
impressions because this client's GSC tracking hadn't started yet" — that check
(`dim_clients.gsc_data_start`) is deferred to the week I bring `dim_clients` into the join.

In [10]:
# Quick anchor for the limitation above: how much of this month's slice
# even has a usable early-window signal, after the >=20 impressions guard in Section 3?
usable_share = len(features) / counts.loc[0, "n_content_items"]
print(f"Content items with usable early-window volume: {len(features):,} "
      f"of {int(counts.loc[0, 'n_content_items']):,} in this month ({usable_share:.1%})")
print("The rest are low/no-impression pages this slice can't say anything reliable about yet.")


Content items with usable early-window volume: 109,592 of 331,437 in this month (33.1%)
The rest are low/no-impression pages this slice can't say anything reliable about yet.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.